# Study report: stimulus-sampling

This notebook is the report template of the study. It reads one instance
of the study (a folder below `results/kernel-latency/`) and shows its
central results, its hypotheses and its plots. The plots are interactive:
hover for values, drag to zoom, click a legend entry to hide a series, and
use the menu above a plot to select the metric.

Commit this file without outputs. The executed copy (with outputs) of an
instance goes into the folder of that instance.

In [ ]:
# Parameters: the instance folder. Set LATBENCH_INSTANCE, or change the default.
import os
import subprocess
from pathlib import Path

INSTANCE = Path(os.environ.get(
    "LATBENCH_INSTANCE",
    "/wrk/z/lab-benchmarks/results/kernel-latency/st-measurement-validity/st-stimulus-sampling@nucleo_g474re"))
REPO = Path(subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True,
                           check=True).stdout.strip())
DESIGN = REPO / "validation/empirical/latencies/st-measurement-validity/st-stimulus-sampling"

In [ ]:
import hashlib
import json
import sys

import pandas as pd
import plotly.graph_objects as go
import yaml

sys.path.insert(0, str(REPO / "scripts/benchmarks"))
import loadmodel  # noqa: E402

pd.set_option("display.max_colwidth", None)


def load(path):
    return json.loads(Path(path).read_text())


study = load(INSTANCE / "study.json")
instance = load(INSTANCE / "instance.json")
design = yaml.safe_load((DESIGN / "study.yaml").read_text())
hypotheses = {h["id"]: h for h in yaml.safe_load((DESIGN / "hypotheses.yaml").read_text())["hypotheses"]}

# The design files must be the files that the instance was built from.
for name, digest in study["design"]["files"].items():
    now = hashlib.sha256((DESIGN / name).read_bytes()).hexdigest()
    if now != digest:
        print(f"WARNING: {name} changed after the build of the instance ({digest[:12]} -> {now[:12]})")

experiments = {e["name"]: load(INSTANCE / e["file"]) for e in study["experiments"]}


def app_data(exp, run_id):
    """The app data file of a run of an experiment of this instance."""
    run = next(r for r in experiments[exp]["runs"] if r["id"] == run_id)
    return load(INSTANCE / f"ex-{exp}" / run["app_json"])


def reference_app_data(ref):
    """The app data file of a reference run (in the folder of its owning instance)."""
    # The instance path has names; the folders have the prefix st- (study) and ex- (experiment).
    root = next(p for p in INSTANCE.parents if not p.name.startswith("st-"))
    folder = root.joinpath(*("st-" + n for n in ref["instance"].split("/")[1:]))
    exp, utc = ref["id"].split("/")
    app = [p for p in (folder / f"ex-{exp}").glob(f"{utc}.*.json") if not p.name.endswith(".run.json")]
    return load(app[0])


print(study["instance"], "| board", study["board"], "| design", study["design"]["safety_rev"][:12])
print()
print(study["question"])

## Central results

In [ ]:
pd.DataFrame([{"id": c["id"], "result": c["text"], "by": c["by"]} for c in study["central_results"]])

## Hypotheses

A hypothesis is one claim about a mechanism. It is wrong if one of its
checks is wrong. The statements and the limits were written before the
runs; the basis of each number is in `hypotheses.yaml` (column `basis`).

In [ ]:
pd.DataFrame([{"id": h["id"], "type": h["type"], "claim": h["statement"], "verdict": h["verdict"],
               "checks confirmed": h["checks_confirmed"]} for h in study["hypotheses"]])

In [ ]:
design_checks = {(h["id"], c["id"]): c for h in hypotheses.values() for c in h["checks"]}
pd.DataFrame([{"check": f"{h['id']} {c['id']}", "measured": c["measured"], "limit": c["limit"],
               "verdict": c["verdict"], "by": c["by"],
               "basis": " ".join(design_checks[(h["id"], c["id"])].get("basis", "").split())}
              for h in study["hypotheses"] for c in h["checks"]])

## Helpers: histogram to CCDF

A histogram of the app has fine bins (width 1 cycle), coarse bins (from
`coarse_from`, width `coarse_width`) and overflow counts by power of two.
The CCDF at x is the share of the samples >= x.

In [ ]:
def ccdf(h):
    """(x, share of samples >= x) at the lower edge of each non-empty bin."""
    edges = [(v, c) for v, c in h["bins"]] + [(v, c) for v, c, _ in h["coarse"]]
    edges += [(2 ** e, c) for e, c in h["ovf_log2"]]
    edges.sort()
    n = h["n"]
    xs, ys, above = [], [], n
    for v, c in edges:
        xs.append(v)
        ys.append(above / n)
        above -= c
    return xs, ys


def ccdf_figure(series, title, metrics):
    """series: {label: {metric: hist}}. A menu selects the metric."""
    fig = go.Figure()
    for m_i, metric in enumerate(metrics):
        for label, hists in series.items():
            x, y = ccdf(hists[metric])
            fig.add_trace(go.Scatter(x=x, y=y, mode="lines", line_shape="hv", name=label,
                                     visible=(m_i == 0), legendgroup=label,
                                     hovertemplate="%{x} cycles: %{y:.2e}<extra>" + label + "</extra>"))
    k = len(series)
    fig.update_layout(
        title=title, xaxis_title="latency (cycles)", yaxis_title="share of samples >= x",
        yaxis_type="log", height=500,
        updatemenus=[{"buttons": [{"label": m, "method": "update",
                                   "args": [{"visible": [j // k == i for j in range(k * len(metrics))]}]}
                                  for i, m in enumerate(metrics)], "x": 1.0, "y": 1.15}])
    return fig

## Tick phase coverage

The share of the edges in each of the 64 phase bins of the tick, against
the uniform share (1 = even). `random` is the stimulus of all
experiments. `fixed-scan` moves the phase by 1 cycle per edge.

In [ ]:
fig = go.Figure()
for exp in ("irq-phase-1m", "wake-phase-1m"):
    run = experiments[exp]["runs"][0]["id"]
    for p in app_data(exp, run)["points"]:
        for metric, ph in p["reps"][0]["phases"].items():
            tot = sum(b[1] for b in ph["bins"])
            fig.add_trace(go.Bar(x=[b[0] for b in ph["bins"]], y=[b[1] * len(ph["bins"]) / tot for b in ph["bins"]],
                                 name=f"{exp} {p['point']} {metric}"))
fig.update_layout(barmode="group", xaxis_title="phase bin (of 64)", yaxis_title="share / uniform share", height=450)
fig.show()

## CCDF by stimulus seed (stim-seed-1m)

Eight stimulus seeds at the same point, with the night run of
`wp3-wake-1m` `baseline-t64` as reference (dashed).

In [ ]:
run = experiments["stim-seed-1m"]["runs"][0]["id"]
series = {p["point"]: p["reps"][0]["hists"] for p in app_data("stim-seed-1m", run)["points"]}
ref = next(r for r in instance["references"] if r["id"].startswith("wp3-wake-1m/"))
night = next(p for p in reference_app_data(ref)["points"] if p["point"] == "baseline-t64")
series["night baseline-t64"] = night["reps"][0]["hists"]
fig = ccdf_figure(series, "stim-seed-1m: CCDF by stimulus seed", ["irq", "isr2thr", "irq2thr"])
for t in fig.data:
    if t.name == "night baseline-t64":
        t.line.dash = "dash"
fig.show()

## CCDF by load seed (load-seed-tail)

The 4M run and three load seeds of 1M samples. The CCDF is a share, so
runs of different length are comparable.

In [ ]:
run = experiments["load-seed-tail"]["runs"][0]["id"]
lst = app_data("load-seed-tail", run)
series = {p["point"]: p["reps"][0]["hists"] for p in lst["points"]}
ccdf_figure(series, "load-seed-tail: CCDF by load seed", ["irq", "isr2thr", "irq2thr"]).show()

## Tail count against the ticks with many expiries

H3 a..c predicted that the count of samples >= 4288 cycles (per 1M
samples) grows with the ticks with >= 3 load timer expiries (from
`loadmodel.py`). The dashed line is that prediction, through the mean of
the 8 stimulus seeds.

In [ ]:
THRESHOLD, K = 4288, 3


def tail_row(label, rep):
    """Ticks with >= K expiries (model) and samples >= THRESHOLD, both per 1M samples."""
    params = loadmodel.log_params({k: str(v) for k, v in rep["meta"].items()})
    ticks = int(params["seconds"] * params["ticks_per_sec"]) + 1
    tset = loadmodel.timers(params["seed"], params["timers"], params["min_ms"], params["max_ms"],
                            params["ticks_per_sec"])
    s = loadmodel.summary(loadmodel.expiries_per_tick(tset, ticks), 1)
    h = rep["hists"]["irq"]
    count = sum(c for v, c in h["bins"] if v >= THRESHOLD) + sum(c for v, c, _ in h["coarse"] if v >= THRESHOLD)
    per_m = 1e6 / h["n"]
    return {"point": label, "load_seed": f"0x{params['seed']:08x}",
            "ticks": loadmodel.at_least(s["counts"], K) * per_m, "samples": count * per_m}


df = pd.DataFrame([tail_row(p["point"], p["reps"][0]) for p in lst["points"]])
seeds = pd.DataFrame([tail_row(p["point"], p["reps"][0])
                      for p in app_data("stim-seed-1m", experiments["stim-seed-1m"]["runs"][0]["id"])["points"]])
x0, y0 = seeds.ticks.mean(), seeds.samples.mean()
fig = go.Figure(go.Scatter(x=df.ticks, y=df.samples, mode="markers+text", text=df.point,
                           textposition="top center", marker_size=10, name="load-seed-tail"))
fig.add_trace(go.Scatter(x=[x0], y=[y0], mode="markers", marker_symbol="diamond", marker_size=10,
                         error_y={"type": "data", "symmetric": False, "array": [seeds.samples.max() - y0],
                                  "arrayminus": [y0 - seeds.samples.min()]},
                         name="stim-seed-1m: mean and range of 8 seeds"))
xs = [0, df.ticks.max() * 1.1]
fig.add_trace(go.Scatter(x=xs, y=[x * y0 / x0 for x in xs], mode="lines", line_dash="dash",
                         name="proportional to the ticks (H3)"))
fig.update_layout(xaxis_title=f"ticks with >= {K} expiries per 1M samples (loadmodel.py)",
                  yaxis_title=f"samples >= {THRESHOLD} cycles per 1M", height=450)
fig.show()
df